In [1]:
from src.evaluation import score_answer
import pickle as pkl
import pandas as pd
import os
from pathlib import Path

dataset_path = Path("~/officeqa/")
data_file = "officeqa.csv"
df = pd.read_csv(os.path.join(dataset_path, data_file))
with open('../results/full_run_new_evolved_final_two.pkl', 'rb') as f:
    data = pkl.load(f)


In [2]:
train_set = {
    'index': [],
    'question': [],
    'predicted': [],
    'ground_truth': [],
    'reasoning': [],
    'score_0.05': [],
    'score_0.01': [],
    'score_0.1': [],
    'score_0.0': [],
    'score_0.025': [],
    'final_score': [],
    'difficulty': [],
    'category': [],
    'source_docs': [],
    'uid': [],
    'source_files': [],
    'agent_trace': []
}

In [3]:
for set_index, example in enumerate(data):
    try:
        question = example.question
        index = example.index
        ground_truth = example.ground_truth
        predicted = example.trace.output.final_answer
        reasoning = example.trace.output.reasoning
        final_score = 0.0
        for tolerance in [0.05, 0.01, 0.1, 0.0, 0.025]:
            curr_score = score_answer(ground_truth, predicted, tolerance)
            final_score += curr_score
            train_set['score_'+str(tolerance)].append(curr_score)
        train_set['final_score'].append(final_score/5)
        train_set['index'].append(index)
        train_set['question'].append(question)
        train_set['predicted'].append(predicted)
        train_set['ground_truth'].append(ground_truth)
        train_set['reasoning'].append(reasoning)
        source_docs = df.iloc[index].source_docs
        source_files = df.iloc[index].source_files
        uid = df.iloc[index].uid
        category = df.iloc[index].pseudo_labels
        difficulty = df.iloc[index].difficulty
        train_set['source_docs'].append(source_docs)
        train_set['source_files'].append(source_files)
        train_set['uid'].append(uid)
        train_set['category'].append(category)
        train_set['difficulty'].append(difficulty)
        train_set['agent_trace'].append(example.trace.messages)
        
        
    except Exception as e:
        print(e)

In [4]:
new_df = pd.DataFrame(train_set)

In [7]:
new_df[new_df['final_score'] < 1.0].to_csv("../ablation_run_incorrect.csv", index=False)